# Subisoform analysis on microglialess Split-seq/Parse data

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "tealeaf").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)


In [ ]:
from pathlib import Path
import pandas as pd

from tealeaf.microglialess import load_all_sublib_inputs
from tealeaf.subisoform_pipeline import run_subisoform_pipeline
from tealeaf.transcript_utils import get_effective_length_weights

In [ ]:
selected_gene_ids = [
    "ENSMUSG00000022748.9",  # Cmss1
    "ENSMUSG00000037965.16", # Zc3h7a
    "ENSMUSG00000092341.5",  # Malat1
]
classes_to_keep = ["EX", "INH", "ASC", "ODC"]

inputs = load_all_sublib_inputs(
    selected_gene_ids=selected_gene_ids,
    classes_to_keep=classes_to_keep,
)

pd.Series(inputs.transcript_ids, name="transcript_id")

In [ ]:
inputs.cell_metadata["class"].value_counts()

In [ ]:
_, transcript_weights = get_effective_length_weights(
    transcript_ids=inputs.transcript_ids,
    fasta_path=Path("/gpfs/commons/groups/knowles_lab/index/salmon/mus_spliceu/spliceu.fa"),
    alias_path=Path("/gpfs/commons/groups/knowles_lab/index/salmon/mus_spliceu/t2t_dedup_v2.tsv"),
)

result = run_subisoform_pipeline(
    cell_ec_matrix=inputs.cell_ec_matrix,
    ec_transcript_mat=inputs.ec_transcript_mat,
    transcript_ids=inputs.transcript_ids,
    transcript_weights=transcript_weights,
    group_labels=inputs.cell_metadata["class"].to_numpy(),
    gtf_filename=Path("/gpfs/commons/groups/knowles_lab/index/kallisto/mus_musculus/with_precursor/gencode.vM32.basic.annotation.gtf"),
    batch_labels=inputs.cell_metadata["sublibrary"].to_numpy(),
    metacell_size=400,
    random_state=0,
    min_samples_per_condition=2,
)

In [ ]:
result.differential_usage[["event_id", "gene_id", "p_value", "fdr", "n_samples"]]

In [ ]:
top_event = result.differential_usage.iloc[0]["event_id"]
result.subisoform_model.subisoform_table.loc[
    result.subisoform_model.subisoform_table["event_id"] == top_event,
    ["subisoform_id", "event_id", "gene_id", "segment_ids", "transcript_ids"],
]

In [ ]:
pd.DataFrame(result.differential_usage.iloc[0]["condition_mean_composition"]).T